# 20.11 — Model compression: quantization

Quantization stores and computes with fewer bits by choosing a scale that maps real weights into a small integer grid.

Numerical linear algebra supplies the scale-and-round operation, while production budgets explain why memory bandwidth can matter as much as parameter count. This notebook rebuilds the scale, clipping, compression, and accuracy-after-quantize calculations as one reusable NumPy method.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2017)


def percentile(values, q):
    return float(np.percentile(np.asarray(values, dtype=float), q))


def softmax(logits, temperature=1.0):
    scaled = np.asarray(logits, dtype=float) / temperature
    shifted = scaled - np.max(scaled, axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values, axis=-1, keepdims=True)


def cross_entropy(target_probs, pred_probs):
    clipped = np.clip(pred_probs, 1e-9, 1.0)
    return float(-np.sum(target_probs * np.log(clipped)))


def expected_calibration_error(confidence, correct, bins=10):
    confidence = np.asarray(confidence, dtype=float)
    correct = np.asarray(correct, dtype=float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    total = len(confidence)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidence >= lo) & (confidence < hi)
        if hi == 1.0:
            mask = (confidence >= lo) & (confidence <= hi)
        if np.any(mask):
            gap = abs(float(np.mean(confidence[mask])) - float(np.mean(correct[mask])))
            ece += float(np.mean(mask)) * gap
    return ece


def make_f17_workload_ladder(seed=2017):
    local_rng = np.random.default_rng(seed)
    return [
        {
            "rung": "D1",
            "name": "tiny hand-check",
            "requests": 60,
            "precision": "int8",
            "scale": 1.8 / 127.0,
            "shape": (8,),
            "load": 1.0,
            "data": np.array([0.73, -0.40, 1.20, -1.70, 0.05, 0.88, -0.91, 1.55]),
        },
        {
            "rung": "D2",
            "name": "clean tensor workload",
            "requests": 400,
            "precision": "int8",
            "scale": 2.5 / 127.0,
            "shape": (64, 32),
            "load": 1.7,
            "data": local_rng.normal(0.0, 0.65, size=(64, 32)),
        },
        {
            "rung": "D3",
            "name": "outliers and bad calibration",
            "requests": 900,
            "precision": "int8",
            "scale": 1.0 / 127.0,
            "shape": (128, 48),
            "load": 2.4,
            "data": np.vstack([
                local_rng.normal(0.0, 0.45, size=(124, 48)),
                local_rng.normal(0.0, 2.2, size=(4, 48)),
            ]),
        },
        {
            "rung": "D4",
            "name": "digits-like classifier weights",
            "requests": 1600,
            "precision": "mixed int8/fp16",
            "scale": 3.0 / 127.0,
            "shape": (10, 64),
            "load": 3.5,
            "data": local_rng.normal(0.0, 0.85, size=(10, 64)),
        },
        {
            "rung": "D5",
            "name": "production-scale load simulation",
            "requests": 5000,
            "precision": "per-channel int8",
            "scale": 4.0 / 127.0,
            "shape": (256, 128),
            "load": 6.0,
            "data": local_rng.normal(0.0, 0.95, size=(256, 128)),
        },
    ]


def preview_ladder(ladder):
    for rung in ladder:
        data = rung.get("data")
        shape = getattr(data, "shape", rung.get("shape"))
        print(f"{rung['rung']} | {rung['name']} | shape={shape} | requests={rung['requests']} | precision={rung['precision']}")
        print("sample:", np.asarray(data).reshape(-1)[:5])


## The concept, built once (D1)

Use the lesson formula $q=\operatorname{round}(x/s)$ and $\hat{x}=sq$. The lesson's symmetric int8 calibration uses max absolute value $1.8$, so $s=1.8/127=0.01417$.

In [ ]:

def quantize_dequantize(x, scale, qmin=-127, qmax=127):
    x = np.asarray(x, dtype=float)
    q = np.round(x / scale)
    q = np.clip(q, qmin, qmax).astype(np.int16)
    x_hat = q.astype(float) * scale
    return q, x_hat

lesson_scale = 1.8 / 127.0
lesson_q, lesson_x_hat = quantize_dequantize(np.array([0.73]), lesson_scale)
lesson_error = abs(float(lesson_x_hat[0]) - 0.73)
float32_mb = 1_000_000 * 4 / 1_000_000
int8_mb = 1_000_000 * 1 / 1_000_000
saved_mb = float32_mb - int8_mb
clipping = 2.2 - 1.8

print("scale", round(lesson_scale, 5))
print("q", int(lesson_q[0]))
print("dequantized", round(float(lesson_x_hat[0]), 3))
print("error", round(lesson_error, 3))
print("saved MB", saved_mb)
print("clipping", clipping)

assert round(lesson_scale, 5) == 0.01417
assert int(lesson_q[0]) == 52
assert round(float(lesson_x_hat[0]), 3) == 0.737
assert round(lesson_error, 3) == 0.007
assert float32_mb == 4.0
assert int8_mb == 1.0
assert saved_mb == 3.0
assert round(clipping, 1) == 0.4


Next, package the arithmetic into a workload scorer. Accuracy-after-quantize is the fraction of values whose dequantized error stays under a tolerance; compression is the float32 byte count divided by the quantized byte count.

In [ ]:

def evaluate_quantized_workload(rung, tolerance=0.03):
    data = np.asarray(rung["data"], dtype=float)
    scale = float(rung["scale"])
    q, x_hat = quantize_dequantize(data, scale)
    errors = np.abs(data - x_hat)
    accuracy_after_quantize = float(np.mean(errors <= tolerance))
    compression = data.size * 4 / q.size
    bandwidth_ms = data.size * 4 / 1_000_000 * rung["load"]
    int8_ms = q.size / 1_000_000 * rung["load"]
    kernel_penalty_ms = 0.04 if rung["precision"] == "int8" else 0.08
    p95_latency_ms = bandwidth_ms * 0.35 + int8_ms + kernel_penalty_ms
    clipped_fraction = float(np.mean(np.abs(data) > scale * 127.0))
    return {
        "accuracy": accuracy_after_quantize,
        "compression": compression,
        "p95_latency_ms": p95_latency_ms,
        "clipped_fraction": clipped_fraction,
        "artifact": x_hat,
    }


## The dataset ladder

The F17 ladder is inline and CPU-only: a tiny hand vector, clean tensors, outlier calibration, a digits-like classifier matrix, and D5 production-scale load/precision simulation.

In [ ]:

ladder = make_f17_workload_ladder()
preview_ladder(ladder)


## Run the SAME method across D1–D5

In [ ]:

quant_results = []
for rung in ladder:
    result = evaluate_quantized_workload(rung)
    result["rung"] = rung["rung"]
    result["name"] = rung["name"]
    quant_results.append(result)

print("rung | accuracy_after_quantize | compression | p95_latency_ms | clipped")
for result in quant_results:
    print(f"{result['rung']} | {result['accuracy']:.3f} | {result['compression']:.1f}x | {result['p95_latency_ms']:.3f} | {result['clipped_fraction']:.3f}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, rung, result in zip(axes[0], ladder, quant_results):
    artifact = np.asarray(result["artifact"]).reshape(-1)[:80]
    ax.plot(artifact, linewidth=1.0)
    ax.set_title(rung["rung"])
    ax.set_ylabel("dequantized")

metric_curve = [result["accuracy"] for result in quant_results]
axes[1, 0].plot([rung["rung"] for rung in ladder], metric_curve, marker="o")
axes[1, 0].set_title("accuracy-after-quantize curve")
axes[1, 0].set_ylim(0.0, 1.05)
for ax, result in zip(axes[1, 1:], quant_results[1:]):
    ax.bar(["latency", "compression", "clip"], [result["p95_latency_ms"], result["compression"], result["clipped_fraction"]])
    ax.set_title(result["rung"] + " operational panel")
plt.tight_layout()


## Pitfall on D5: forgetting calibration data

Bad calibration can make int8 look small but inaccurate. Reproduce one global scale that clips channels, then fix it with representative per-channel calibration.

In [ ]:

d5 = ladder[-1]
d5_data = np.asarray(d5["data"], dtype=float)
bad_scale = 0.9 / 127.0
bad_q, bad_hat = quantize_dequantize(d5_data, bad_scale)
bad_error = np.abs(d5_data - bad_hat)
bad_accuracy = float(np.mean(bad_error <= 0.03))
bad_clipped = float(np.mean(np.abs(d5_data) > bad_scale * 127.0))

channel_max = np.maximum(np.max(np.abs(d5_data), axis=1, keepdims=True), 1e-6)
channel_scale = channel_max / 127.0
good_q = np.round(d5_data / channel_scale)
good_q = np.clip(good_q, -127, 127).astype(np.int16)
good_hat = good_q.astype(float) * channel_scale
good_error = np.abs(d5_data - good_hat)
good_accuracy = float(np.mean(good_error <= 0.03))
good_clipped = float(np.mean(np.abs(d5_data) > channel_scale * 127.0))

print("bad accuracy", round(bad_accuracy, 3), "bad clipped", round(bad_clipped, 3))
print("fixed accuracy", round(good_accuracy, 3), "fixed clipped", round(good_clipped, 3))
assert good_accuracy > bad_accuracy
assert good_clipped <= bad_clipped


## Evaluate it + Practice

- Metric: track accuracy-after-quantize plus compression; compare with the no-skill baseline `float32 model with 1.0x compression`.
- Cheap sanity check: dequantizing q=52 should return about 0.737.
- Ablation: force a bad global scale and watch accuracy drop.
- Failure signals: high clipped fraction or accuracy that falls on D5.
- Production guardrail: monitor the metric by rung instead of averaging D1 with D5.

Practice 1: Change one D5 load or precision setting and predict the metric before running.

Practice 2: Add one operational constraint, such as memory budget or p95 latency budget, and mark the first rung that violates it.

Practice 3: Write a one-line launch rule that uses the metric and one pitfall guardrail.